# 06 — Satellite Stress Classifier (ResNet50) — Colab Training

Trains the **only NEW model** in the multi-modal pipeline.  YOLOv8 (field) and the LSTM+XGBoost (features) are FROZEN — do NOT retrain them here.

**Inputs**
- PlanetScope chips under `data/visual/planetscope/<class>/...`  *(once Education key arrives)*
- Sentinel-2 stand-in chips under `data/visual/planetscope/s2_standin/<class>/...`

**Output**
- `models/visual/satellite_cnn_resnet50.pt` — state-dict
- `models/visual/satellite_cnn_metrics.json` — accuracy, per-class F1, confusion matrix

**Classes (3)**: `healthy`, `mild_stress`, `severe_stress`.  Disease is intentionally NOT a class — leaf-scale disease is the field modality's job.


In [ ]:
# Colab bootstrap (skip if running locally with venv)
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Trak-AI_KDS'
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
!pip -q install torch torchvision pillow scikit-learn

In [ ]:
import torch, torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from pathlib import Path
import json, hashlib, time
from sklearn.metrics import classification_report, confusion_matrix

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', DEVICE)
DATA_DIR = Path('data/visual/planetscope')          # update once classes folders exist
OUT_WEIGHTS = Path('models/visual/satellite_cnn_resnet50.pt')
OUT_METRICS = Path('models/visual/satellite_cnn_metrics.json')
OUT_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize(256), transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full = datasets.ImageFolder(DATA_DIR, transform=train_tf)
n_total = len(full)
n_train = int(0.75 * n_total); n_val = int(0.15 * n_total); n_test = n_total - n_train - n_val
train_set, val_set, test_set = random_split(full, [n_train, n_val, n_test], generator=torch.Generator().manual_seed(42))
val_set.dataset.transform = eval_tf; test_set.dataset.transform = eval_tf

train_dl = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2)
val_dl   = DataLoader(val_set,   batch_size=32, shuffle=False, num_workers=2)
test_dl  = DataLoader(test_set,  batch_size=32, shuffle=False, num_workers=2)
print('classes:', full.classes, 'sizes:', n_train, n_val, n_test)

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, len(full.classes))
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optim = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=10)

EPOCHS = 10
best_val = 0.0
for ep in range(EPOCHS):
    model.train(); t0 = time.time()
    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optim.zero_grad()
        loss = criterion(model(x), y); loss.backward(); optim.step()
    sched.step()
    # Validation
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item(); total += y.size(0)
    acc = correct / total if total else 0.0
    print(f'epoch {ep+1}/{EPOCHS} val_acc={acc:.3f} t={time.time()-t0:.1f}s')
    if acc > best_val:
        best_val = acc
        torch.save(model.state_dict(), OUT_WEIGHTS)

In [ ]:
# Test set evaluation + metrics dump
model.load_state_dict(torch.load(OUT_WEIGHTS, map_location=DEVICE))
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x, y in test_dl:
        x = x.to(DEVICE)
        y_pred.extend(model(x).argmax(1).cpu().tolist())
        y_true.extend(y.tolist())
report = classification_report(y_true, y_pred, target_names=full.classes, output_dict=True, zero_division=0)
cm = confusion_matrix(y_true, y_pred).tolist()

h = hashlib.sha256(OUT_WEIGHTS.read_bytes()).hexdigest()
metrics = {
    'classes': full.classes,
    'best_val_accuracy': best_val,
    'test_report': report,
    'confusion_matrix': cm,
    'model_sha256': h,
}
OUT_METRICS.write_text(json.dumps(metrics, indent=2))
print('saved metrics ->', OUT_METRICS)
print('sha256       =', h)

## Hand-off to the visual_validation module

After training:
1. Copy `models/visual/satellite_cnn_resnet50.pt` from Colab Drive to the repo (if local).
2. The consensus engine will auto-detect the file via `models.satellite_cnn.load_predictor()` and start emitting the satellite modality vote.
3. Record the SHA-256 above in `docs/REPRODUCIBILITY.md` under the new model section.
